In [27]:
import os
import cv2
import time
import torch
import datetime
import numpy as np
from torch import nn

from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

from pytorch_grad_cam import GradCAM, \
    HiResCAM, \
    ScoreCAM, \
    GradCAMPlusPlus, \
    AblationCAM, \
    XGradCAM, \
    EigenCAM, \
    EigenGradCAM, \
    LayerCAM, \
    FullGrad, \
    FinerCAM, \
    KPCACAM, \
    AlignGradCAM,\
    GradCAMElementWise

from pytorch_grad_cam import GuidedBackpropReLUModel
from pytorch_grad_cam.ablation_cam_new import AblationCAMNEW
from pytorch_grad_cam.grad_cam_new import Grad_CAM_NEW
from pytorch_grad_cam.layer_cam_new import Layer_CAM_NEW
from pytorch_grad_cam.score_cam_new import ScoreCAMNew
from pytorch_grad_cam.mgrad_cam import MGradCAM

from pytorch_grad_cam.utils.image import show_cam_on_image, \
    deprocess_image, \
    preprocess_image, show_image

from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, ClassifierOutputSigmoidTarget

In [28]:
methods = {"gradcam": GradCAM,
        "hirescam": HiResCAM,
        "scorecam": ScoreCAM,
        "scorecamnew": ScoreCAMNew,
        "gradcam++": GradCAMPlusPlus,
        "ablationcam": AblationCAM,
        "ablationcamnew": AblationCAMNEW,
        "xgradcam": XGradCAM,
        "eigencam": EigenCAM,
        "eigengradcam": EigenGradCAM,
        "layercam": LayerCAM,
        "fullgrad": FullGrad,
        "gradcamelementwise": GradCAMElementWise,
        "layercamnew": Layer_CAM_NEW,
        "gradcamnew": Grad_CAM_NEW,
        "finercam": FinerCAM,
        "kpcacam": KPCACAM,
        "mgradcam": MGradCAM,
        "aligngradcam": AlignGradCAM,}

In [29]:
coco_class = ['person','bicycle','car','motorcycle', 'airplane','bus',
'train','truck',
'boat',
'traffic light',
'fire hydrant',
'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
'carrot',
 'hot dog',
'pizza',
 'donut',
 'cake',
 'chair',
'couch',
'potted plant',
'bed',
'dining table',
'toilet',
 'tv',
'laptop', 'mouse',
'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
'sink',
'refrigerator',
 'book',
 'clock',
'vase',
 'scissors',
 'teddy bear',
 'hair drier',
'toothbrush'
]

In [30]:
path = r"/devdata/home/homefun"

In [31]:
target = [70,81]
image_path = os.path.join(path, r"DATA/coco_cam/train2017/000000020652.jpg")
save_path = os.path.join(path, r"CAM-copy/pic_result_new/coco_InsDel")
resume = os.path.join(path, r"weights/coco_cam/coco_loss_20250901193927/best_acc.pth")
aug_smooth = False
eigen_smooth = False
use_cuda = True if torch.cuda.is_available()  else False
device = torch.device("cuda:0") if torch.cuda.is_available()  else torch.device("cpu")

In [32]:
def build_coco_id_mappings():
    """
    返回:
      coco_id_to_contiguous: dict[int->int], 1..90 -> 0..79（跳过空洞）
      contiguous_to_coco_id: dict[int->int], 0..79 -> 1..90
    """
    # 建立一个映射表(1-90)---> (0-79)
    unused_ids = [12, 26, 29, 30, 45, 66, 68, 69, 71, 83]

    coco_id_to_contiguous = {}
    contiguous_to_coco_id = {}
    contiguous_id = 0
    for coco_id in range(1, 91):
        if coco_id in unused_ids:
            continue
        coco_id_to_contiguous[coco_id] = contiguous_id
        contiguous_to_coco_id[contiguous_id] = coco_id
        contiguous_id += 1

    assert contiguous_id == 80, f"Expected 80 classes, got {contiguous_id}"
    return coco_id_to_contiguous, contiguous_to_coco_id

In [33]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, 80)
model.load_state_dict(torch.load(resume, map_location='cpu')['model'])

# target_layers = [model.layer4]
# target_layers = [model.layer2, model.layer3, model.layer4]

# =======================
# COCO ID <-> 连续 ID 映射
# =======================
coco_id_to_contiguous, contiguous_to_coco_id = build_coco_id_mappings()
# ---- 映射 targets（COCO -> 连续）----
mapped_targets = []
for c in target:
    if c in coco_id_to_contiguous:
        mapped_targets.append(coco_id_to_contiguous[c])

rgb_img = cv2.imread(image_path)[:, :, ::-1]
rgb_img = cv2.resize(rgb_img, [224, 224])
rgb_img = np.float32(rgb_img) / 255
input_tensor = preprocess_image(rgb_img,
                                mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
targets = [ClassifierOutputTarget(int(tar)) for tar in mapped_targets]
input_tensor = torch.cat([input_tensor for tar in mapped_targets], dim=0)

model, input_tensor = model.eval().to(device), input_tensor.to(device)
y = torch.sigmoid(model(input_tensor))[0, mapped_targets]
print(torch.sigmoid(model(input_tensor))[0, mapped_targets])

In [34]:
def process(method = "layercam", target_layers = [model.layer4]):
    cam_algorithm = methods[method]
    cam = cam_algorithm(model=model, target_layers=target_layers, use_cuda=use_cuda)
    # AblationCAM and ScoreCAM have batched implementations.
    # You can override the internal batch size for faster computation.
    cam.batch_size = 32
    # Finer-CAM的时候需要用到device，其它CAM类方法可以注释掉
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if hasattr(cam, 'device'):
        cam.device = device
    else:
        # 如果CAM没有device属性，手动设置
        cam.device = device
    # 对于某些CAM方法，可能需要手动设置activations_and_gradients的设备
    if hasattr(cam, 'activations_and_grads'):
        cam.activations_and_grads.device = device
    grayscale_cams = cam(input_tensor=input_tensor,
                            targets=targets,
                            aug_smooth=aug_smooth,
                            eigen_smooth=eigen_smooth)
    return grayscale_cams
    

In [35]:
def getauc(p, index, mapped_targets):
    # print(os.path.join(path, 'weights', p, 'record.csv'))
    csv = pd.read_csv(os.path.join(path, 'weights', p, 'record.csv'))
    result = []
    # print(csv.columns)
    if index == 'delete':
        percentiles = [i for i in range(100, 0, -1)]
    else:
        percentiles = [i for i in range(99, -1, -1)]
    for i in percentiles:
        result.append(csv.loc[(csv['image_id'] == image_path.split('/')[-1]) 
                              & (csv['label'] == mapped_targets), index + '_' + str(i)].values[0])
    return result

In [36]:
def savepic(grayscale_cams, method, layers):
    if not os.path.exists(os.path.join('coco_InsDel', method)):
        os.makedirs(os.path.join('coco_InsDel', method))
    for index, grayscale_cam in enumerate(grayscale_cams):
        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        image = Image.fromarray(cam_image.astype(np.uint8))
        image.save(os.path.join('coco_InsDel', method,
                                image_path.split('/')[-1].split('.')[0] + '_' + layers + '_' + coco_class[mapped_targets[index]] + '.pdf'), 'PDF', resolution=100.0, save_all=True)
        image.save(os.path.join('coco_InsDel', method,
                                image_path.split('/')[-1].split('.')[0] + '_' + layers + '_' + coco_class[mapped_targets[index]] + '.png'))
    image = Image.fromarray((rgb_img * 255).astype(np.uint8))
    image.save(os.path.join('coco_InsDel', method,
                                 image_path.split('/')[-1].split('.')[0] + '.pdf'), 'PDF', resolution=100.0, save_all=True)
    image.save(os.path.join('coco_InsDel', method,
                                 image_path.split('/')[-1].split('.')[0] + '.png'))

In [37]:
def compare(camdict, target_layers = [model.layer4]):
    recordDict = {}
    for method, p in camdict.items():
        grayscale_cams = process(method, target_layers)
        savepic(grayscale_cams, method, '432' if len(target_layers) > 1 else '4')
    for t in mapped_targets:
        insdict = {}
        deldict = {}
        for method, p in camdict.items():
            insdict[method] = getauc(p, 'insert', t)
            deldict[method] = getauc(p, 'delete', t)
        print(len(insdict), len(deldict))
        recordDict[t] = {'ins' : insdict, 'del' : deldict}
    return recordDict

In [38]:
def main(method = "layercam", target_layers = [model.layer4], save=False, save_name='432'):
    grayscale_cams = process(method, target_layers)
    # Here grayscale_cam has only one image in the batch
    for index, grayscale_cam in enumerate(grayscale_cams):
        plt.subplot(2, 2, index + 1)
        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        if save:
            image = Image.fromarray(cam_image.astype(np.uint8))
            image.save(os.path.join('coco_InsDel', method,
                        image_path.split('/')[-1].split('.')[0] + '_' + save_name + '_' + coco_class[mapped_targets[index]] + '.png'))
        plt.imshow(cam_image)
    plt.tight_layout()
    plt.show()

In [39]:
camdict = { 
            'gradcam'   : r"COCO_CAM/gradcam/4/202511261122",            
            # 'aligngradcam' : r"CAM/aligngradcam/4/202510171543",
            'gradcam++' : r"COCO_CAM/gradcam++/4/202511270930",
            'layercam'  : r"COCO_CAM/layercam/4/202511270925",
            'fullgrad'  : r"COCO_CAM/fullgrad/4/202511271844",
            # # 'mgradcam'  : r"CAM/mgradcam/4/202312062043",
            # 'scorecam'  : r"COCO_CAM/scorecam/4/202509172029",
            'ablationcam'  : r"COCO_CAM/ablationcam/4/202511271729",
            # 'xgradcam'  : r"CAM/xgradcam/4/202508311748",
            # 'eigencam'  : r"CAM/eigencam/4/202509171726",
            # 'kpcacam'  : r"COCO_CAM/kpcacam/4/202509171449",
            'finercam'  : r"COCO_CAM/finercam/4/202511281154",
            'mgradcam'  : r"COCO_CAM/mgradcam/432/202511262312",
            }
recordDict = compare(camdict)
recordcsv = pd.DataFrame(columns=list(range(103)))
for k, v in recordDict.items():
    for k1, v1 in v['del'].items():
        l = ['del', coco_class[k], k1]
        l.extend(v1)
        recordcsv.loc[len(recordcsv)] = l
for k, v in recordDict.items():
    for k1, v1 in v['ins'].items():
        l = ['ins', coco_class[k], k1]
        l.extend(v1)
        recordcsv.loc[len(recordcsv)] = l

save_dir = "coco_InsDel"
save_file = os.path.join(save_dir, "auc.xlsx")
os.makedirs(save_dir, exist_ok=True)

print("当前工作目录:", os.getcwd())
print("recordDict keys:", recordDict.keys())
print("recordcsv shape:", recordcsv.shape)
print("将要保存的路径:", os.path.abspath(save_file))

recordcsv.to_excel(save_file, index=False)
print("✅ 保存成功:", os.path.abspath(save_file))

curve_dir = os.path.join(save_dir, "curves")
os.makedirs(curve_dir, exist_ok=True)

for t, results in recordDict.items():
    for metric in ["ins", "del"]:
        plt.figure(figsize=(6, 4))
        colors = plt.cm.get_cmap('tab20', len(results[metric]))
        for idx, (method, values) in enumerate(results[metric].items()):
            plt.plot(range(len(values)), values, label=method, color=colors(idx))
        # plt.xlabel(f"{metric.upper()} point")
        # plt.ylabel("Class Score")
        plt.title(f"{coco_class[t]}")
        plt.legend()
        out_path = os.path.join(curve_dir, f"{metric}_{coco_class[t]}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"✅ 曲线已保存: {os.path.abspath(out_path)}")

# recordcsv.to_excel( os.path.join('coco_InsDel', 'auc.xlsx'), index=False)

In [40]:
main("mgradcam", [model.layer2, model.layer3, model.layer4])

In [41]:
main("gradcam")

In [42]:
main("gradcam++")

In [43]:
main("layercam")

In [44]:
main("fullgrad")

In [45]:
main("finercam")

In [46]:
main("ablationcam")